# Demo: Track Annotation Pipeline

End-to-end demo của module `track_annotation` trên một video Bradford Bulls.

Notebook này:
1. Setup môi trường
2. Load config
3. Run pipeline (detect + track + keyframe + clip + package)
4. Inspect kết quả
5. Hướng dẫn launch reviewer UI

Yêu cầu: GPU CUDA, weights `weights/yolo11l.pt`, 1 video trong `data/videos/`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Add v3/src to path so we can import track_annotation
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from track_annotation.config import load_config
from track_annotation.pipeline.package_builder import build_package
from track_annotation.utils.logging import setup_logging

setup_logging(level='INFO', log_to_file=False)
print('OK')

## 2. Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'CUDA: {torch.cuda.get_device_name(0)} ({torch.cuda.device_count()} device)')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print('MPS (Apple Silicon)')
else:
    print('!!! Using CPU; pipeline will be very slow')

## 3. Load config

In [ ]:
cfg = load_config(ROOT / 'configs' / 'person_tracking.yaml')

# For demo: limit to first 60 seconds
cfg.video.max_duration_s = 60

print(f'Device         : {cfg.resolve_device()}')
print(f'Detection conf : {cfg.detection.conf}')
print(f'Tracker        : {cfg.tracking.tracker}')
print(f'Processing FPS : {cfg.video.processing_fps}')
print(f'Max duration   : {cfg.video.max_duration_s}s')

## 4. Run pipeline

In [ ]:
# UPDATE THIS to a video that exists in data/videos/
VIDEO = ROOT / 'data' / 'videos' / 'YOUR_VIDEO.mp4'
OUTPUT = ROOT / 'data' / 'annotation_packages' / 'demo_run'

assert VIDEO.exists(), f'Update VIDEO path; file not found: {VIDEO}'

package_path = build_package(VIDEO, OUTPUT, cfg)
print(f'\nPackage written to: {package_path}')

## 5. Inspect kết quả

In [ ]:
import json

manifest = json.loads((OUTPUT / 'manifest.json').read_text())

print(f'Video         : {manifest["video"]["filename"]}')
print(f'Resolution    : {manifest["video"]["width"]}x{manifest["video"]["height"]}')
print(f'Duration      : {manifest["video"]["duration_s"]:.1f}s')
print()
print(f'Total tracks  : {manifest["stats"]["num_tracks"]}')
print(f'Total dets    : {manifest["stats"]["total_detections"]}')
print(f'Mean track len: {manifest["stats"]["mean_track_duration_s"]:.2f}s')

In [ ]:
# Preview keyframes của track đầu tiên
from IPython.display import Image, display

tracks = sorted((OUTPUT / 'tracks').iterdir())
if tracks:
    track_dir = tracks[0]
    print(f'Track: {track_dir.name}')
    for img in sorted(track_dir.glob('keyframe_*_crop.jpg')):
        print(f'  {img.name}')
        display(Image(str(img)))
else:
    print('No tracks found')

## 6. Launch reviewer UI

Trong terminal:

```bash
bash scripts/run_reviewer.sh data/annotation_packages/demo_run
```

Mở trình duyệt http://localhost:8501 để bắt đầu annotate.

Sau khi annotate xong, export sang YOLO format:

```bash
python -m track_annotation.cli export \
    --package data/annotation_packages/demo_run \
    --format yolo \
    --output data/yolo_dataset \
    --single-class
```